# DeepLOB "Super Model" (DeepLOB Encoder + Seq2Seq Decoder)

This notebook implements the architecture inspired by the MHF paper.

**Architecture:**
1.  **Encoder:** DeepLOB (Convolutional Blocks + Inception Module + LSTM) to extract rich spatial features from the LOB.
2.  **Decoder:** Seq2Seq LSTM with Additive Attention to generate the 60-second midprice path step-by-step.


In [1]:
import numpy as np
from pathlib import Path
import numpy as np
import torch
import random
from utils.utils import *
from data.simulate_walk_the_book import *
from utils.datastuff import TrainCfg
from utils.train import train_val


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Fix randomness for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


device: cpu


In [5]:
# Paths and volume_to_fill
root = Path("data") if Path("data").exists() else Path.cwd()
import sys
if str(root / "src") not in sys.path:
    sys.path.append(str(root / "src"))

symbol_dir = root / "BTCUSDT"
X_path = symbol_dir / "X_train.parquet"
Y_path = symbol_dir / "y_train.parquet"
vol_path = symbol_dir / "vol_to_fill.txt"

volume_to_fill = None
if vol_path.exists():
    import re
    with open(vol_path) as f:
        m = re.search(r"([\d.]+)", f.read())
    if m:
        volume_to_fill = float(m.group(1))
print("volume_to_fill:", volume_to_fill)


volume_to_fill: 4.0


In [6]:
# DeepLOB only take LOB features as input
LOB_COLS = []
for i in range(1, 6):
    LOB_COLS.append(f"ask_price_{i}")
    LOB_COLS.append(f"ask_vol_{i}")
    LOB_COLS.append(f"bid_price_{i}")
    LOB_COLS.append(f"bid_vol_{i}")

FEATURE_COLS = LOB_COLS

# Verification print
print(f"CNN Input Width: 4 (Columns: Price/Vol/Price/Vol)")
print(f"CNN Input Height: 5 (Rows: Levels 1 through 5)")
print(f"Total Features: {len(FEATURE_COLS)}")

CNN Input Width: 4 (Columns: Price/Vol/Price/Vol)
CNN Input Height: 5 (Rows: Levels 1 through 5)
Total Features: 20


In [7]:
# --- Execution ---
config = TrainCfg(

    # Hyperparameters
    epochs = 1,#20,
    batch_size = 32,
    lr = 1e-3,
    weight_decay = 1e-5,
    smooth_lambda = 0.01,
    
    # Windows
    input_window = 300,   # Look-back
    target_window = 60,   # Horizon
    val_ratio = 0.2,

    # Environment
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    
    # Features & Data
    x_path = X_path,
    y_path = Y_path,
    feature_cols = FEATURE_COLS,
)

model, scalers, val_loader, va_id_map = train_val(config)

Epoch 01 | Train: 0.618875 | Val: 374009.476103


In [19]:
from utils.test import generate_test_predictions

# --- Execution ---
test_preds, test_id_map = generate_test_predictions(model, config, num_ids=5)

Loading test data...
Sampling first 5 IDs for inference...
Preprocessing test data...
Running inference on 5 instruments...


In [20]:
test_preds

array([[0.459546  , 0.4202756 , 0.41500264, 0.41788557, 0.42206392,
        0.4255798 , 0.4281038 , 0.4297807 , 0.43084148, 0.43148807,
        0.43186912, 0.43208575, 0.43220374, 0.43226436, 0.43229264,
        0.4323035 , 0.43230546, 0.43230337, 0.43229985, 0.43229607,
        0.43229267, 0.43228987, 0.43228754, 0.43228588, 0.43228456,
        0.43228367, 0.43228295, 0.43228248, 0.43228212, 0.43228185,
        0.4322817 , 0.4322816 , 0.43228152, 0.43228146, 0.4322814 ,
        0.4322814 , 0.43228137, 0.43228137, 0.43228135, 0.43228135,
        0.43228135, 0.43228135, 0.43228135, 0.43228135, 0.43228135,
        0.43228135, 0.43228135, 0.43228135, 0.43228135, 0.43228135,
        0.43228135, 0.43228135, 0.43228135, 0.43228135, 0.43228135,
        0.43228135, 0.43228135, 0.43228135, 0.43228135, 0.43228135],
       [0.42050347, 0.36994514, 0.37367922, 0.38797203, 0.40187147,
        0.41269678, 0.42041764, 0.4256807 , 0.42917266, 0.43144795,
        0.43291083, 0.43384144, 0.43442807, 0.4

* **The Map:** This `test_id_np` array pairs the model's row index (e.g., Row 5) with the actual asset ID (e.g., BTC).

In [21]:
test_id_np = np.array([(idx, uid.item()) for idx, uid in test_id_map], dtype=np.uint64)

# Create a quick summary for inspection
print(f"{'Anonymized ID':<25} | {'First Pred (t+1)':<18} | {'Last Pred (t+60)':<18}")
print("-" * 70)

for i in range(len(test_id_np)):  # Look at first 5
    orig_id = test_id_np[i, 1]
    first_p = test_preds[i, 0]
    last_p  = test_preds[i, -1]
    print(f"{str(orig_id):<25} | {first_p:18.6f} | {last_p:18.6f}")

Anonymized ID             | First Pred (t+1)   | Last Pred (t+60)  
----------------------------------------------------------------------
349798595607087602        |           0.459546 |           0.432281
2362602560156466613       |           0.420503 |           0.435370
5456753287518378172       |           0.405972 |           0.423917
7807403476349746613       |           0.465817 |           0.450775
14310251273089688238      |           0.459802 |           0.428046
